# 🎒 Test: ibm-granite/granite-docling-258M OCR

Dieses Notebook dient dazu, das lokale Vision-Language-Modell **ibm-granite/granite-docling-258M** Schritt für Schritt auf Ihrer Maschine auszuführen. Dadurch können Sie eventuelle Fehlermeldungen (z. B. Out of Memory, Treiber-Probleme oder fehlende Bibliotheken) direkt isolieren und einsehen.

## 1. Imports & System-Check
Hier prüfen wir, ob PyTorch, Transformers und eventuell CUDA (Grafikkarten-Unterstützung) zur Verfügung stehen.

In [ ]:
import sys
import os
import torch
import transformers
from PIL import Image
import io

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"CUDA (GPU) verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Gerätename: {torch.cuda.get_device_name(0)}")

## 2. Modell & Prozessor laden
Hier laden wir das Modell von Hugging Face herunter (falls noch nicht geschehen) und initialisieren es. 
Falls hier ein Fehler auftritt (z. B. Speicherplatzmangel oder Treiberprobleme), wird die genaue Ursache ausgegeben.

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_id = "ibm-granite/granite-docling-258M"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Lade Modell auf Gerät: {device}...")

try:
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")
    print("✅ Modell und Prozessor erfolgreich geladen!")
except Exception as e:
    print(f"❌ Fehler beim Laden des Modells: {e}")

## 3. Test-Bild vorbereiten
Wir erstellen ein einfaches Testbild mit Text, um die Erkennung zu überprüfen. Alternativ können Sie unten den Pfad zu einem echten Foto eintragen.

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "image"}, {"type": "text", "text": "Convert this page to docling."}]
}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
print("Generiere Text...")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=1024)
prompt_length = inputs["input_ids"].shape[-1]
generated_text = processor.decode(outputs[0][prompt_length:], skip_special_tokens=False).lstrip()
print("--- Roher VLM-Output ---")
print(generated_text)


## 4. OCR Generierung ausführen
Hier senden wir das Bild an das Granite-Modell und bitten es, den Inhalt auszulesen.

In [ ]:
prompt = "Convert this page to docling."
inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

print("Generiere Text...")
try:
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024)
    generated_text = processor.decode(outputs[0], skip_special_tokens=True)
    print("✅ Generierung abgeschlossen!")
    print("--- Roher VLM-Output ---")
    print(generated_text)
except Exception as e:
    print(f"❌ Fehler bei der Generierung: {e}")

## 5. Parsing & Strukturierung testen
Zuletzt jagen wir den erkannten Text durch unsere Parsing-Funktion (aus `parser.py`), die die Mengen und Artikeltexte extrahiert.

In [ ]:
# Granite Docling – reparierte direkte Ausführung
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_id = "ibm-granite/granite-docling-258M"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")

messages = [{
    "role": "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": "Extrahiere den ganzen Text des Bildes."}
    ]
}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=800)
prompt_length = inputs["input_ids"].shape[-1]
generated_text = processor.decode(outputs[0][prompt_length:], skip_special_tokens=False).lstrip()
print("--- Granite Docling Output ---")
print(generated_text)
